# Cleaning Downloaded Data from avian-flu

Author: Alexander Maksiaev

Purpose: Clean downloaded data from avian-flu, rename sequences according to convention, de-duplicate from GISAID

In [1]:
# Housekeeping

import os
import glob 
import pandas as pd
import xml.etree.ElementTree as ET
import requests
import time
import numpy as np
import dateutil 
from datetime import datetime, timedelta
from collections import defaultdict 
import importlib
import utils  
importlib.reload(utils)
from utils import * 

# home = "C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu_Files/"
home = "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu"
# downloads = "C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu_Files/"
downloads = "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/"
originals = downloads + "Andersen_Downloads/"
temp_files = downloads + "Andersen_Temp_Files/"
complete_files = downloads + "Andersen_Complete_Files/"

os.chdir(downloads)

In [19]:
# Read metadata

metadata_folder = originals + "avian-influenza/metadata/"
os.chdir(metadata_folder)

metadata = pd.read_csv("SraRunTable_automated.csv")

# print(len(metadata)) # 7397 rows

# Split date format to only check year
for date in metadata["Collection_Date"]:
    if "/" in date or date == "missing":
        metadata = metadata[metadata["Collection_Date"] != date]
metadata["Collection_Date"] = metadata["Collection_Date"].apply(lambda x: x.split("-")[0])
metadata["Collection_Date"] = metadata["Collection_Date"].apply(lambda x: int(x))

# Find only >= 2024 using run ID from metadata
metadata_new = metadata[metadata["Collection_Date"] >= 2024]
metadata_new["Collection_Date"] = metadata_new["Collection_Date"].astype(int) # Years are not floats

# Find only >= last date using Release Date from metadata (2025, 4, 1)
metadata_new["ReleaseDate"] = metadata_new["ReleaseDate"].apply(lambda x: dateutil.parser.parse(x).strftime("%Y-%m-%d"))
metadata_new = metadata_new[metadata_new["ReleaseDate"] >= datetime(2024, 1, 1).strftime("%Y-%m-%d")]

print(len(metadata_new)) # 6053 rows between 1/1/2024 and 3/31/2025

C:\Users\maksiaevai.NCBI_NT\AppData\Local\Temp\1\ipykernel_19100\80415036.py:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  metadata_new["Collection_Date"] = metadata_new["Collection_Date"].astype(int) # Years are not floats


7044


C:\Users\maksiaevai.NCBI_NT\AppData\Local\Temp\1\ipykernel_19100\80415036.py:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  metadata_new["ReleaseDate"] = metadata_new["ReleaseDate"].apply(lambda x: dateutil.parser.parse(x).strftime("%Y-%m-%d"))


### Naming convention ###
>A/[host]/[geo_loc_name]/[isolate]/[year]|[serotype: H5N1]|[collection_date]|[host_type]|[genotype: B3.13 or D1.1]

host_type is from manual animal reference

In metadata, we have: host, geo_loc_name, isolate, year

We need: geo_loc_name, collection_date, host_type, genotype

host = Host

geo_loc_name (primary) = geo_loc_name

geo_loc_name (secondary) = genbank_mapping.tsv > genbank_name

isolate = isolate

collection date (primary) = Collection_Date

collection date (secondary) = https://www.ncbi.nlm.nih.gov/genbank/ > BioSample (input: BioSample) > Nucleotide > [first result] > collection_date

serotype = serotype

host type = [from ref] 

genotype = [from genoflu] -- use genoflu_results.tsv

In [20]:
# # Get genotype from genoflu
# os.chdir(temp_files)
# output_tsv = pd.read_csv("output.tsv", delimiter="\t")

# b313_and_d11_only = output_tsv[(output_tsv["Genotype"] == "B3.13") | (output_tsv["Genotype"] == "D1.1")]
# b313_and_d11_only = b313_and_d11_only.rename(columns={"sample": "Run"})
# b313_and_d11_only = b313_and_d11_only.drop_duplicates(subset="Run", keep="last")
# # print(b313_and_d11_only)
# print(len(b313_and_d11_only)) # 5160 rows

# metadata_new = metadata_new.merge(b313_and_d11_only, on="Run", how="inner")

# print(len(metadata_new)) # 5160

In [21]:
# Get genotype from genoflu_results.tsv

output_tsv = pd.read_csv("genoflu_results.tsv", delimiter="\t")

# b313_and_d11_only = output_tsv[(output_tsv["Genotype"] == "B3.13") | (output_tsv["Genotype"] == "D1.1")]
# b313_and_d11_only = b313_and_d11_only.rename(columns={"sample": "Run"})
# b313_and_d11_only = b313_and_d11_only.drop_duplicates(subset="Run", keep="last")
# # print(b313_and_d11_only)
# print(len(b313_and_d11_only)) 

# metadata_new = metadata_new.merge(b313_and_d11_only, on="Run", how="inner")

metadata_new["Genotype"] = output_tsv["Genotype"]
metadata_new = metadata_new[~metadata_new["Genotype"].str.contains('Not assigned')]

print(len(metadata_new)) 

6699


In [22]:
print(len(metadata_new))

print(metadata_new)

6699
              Run Assay Type  AvgSpotLen      Bases    BioProject  \
1     SRR28752447        WGS      241.29   86080323  PRJNA1102327   
2     SRR28752448        WGS      250.30   75035343  PRJNA1102327   
3     SRR28752449        WGS      146.61   59363690  PRJNA1102327   
4     SRR28752450        WGS      251.31  119232569  PRJNA1102327   
7     SRR28752453        WGS      144.71   37055919  PRJNA1102327   
...           ...        ...         ...        ...           ...   
8383  SRR33029748        WGS      147.39  252799915   PRJNA980729   
8384  SRR33029749        WGS      147.88  167615693   PRJNA980729   
8385  SRR33029750        WGS      146.73   95329644   PRJNA980729   
8386  SRR33029751        WGS      146.12  269328456   PRJNA980729   
8387  SRR33029752        WGS      145.33  270721593   PRJNA980729   

         BioSample BioSampleModel     Bytes Center Name  Collection_Date  ...  \
1     SAMN41019237          Viral  28164109   USDA-NVSL             2024  ...   
2   

In [23]:
# Get geolocation from genbank_mapping.tsv

genbank_mapping = pd.read_csv("genbank_mapping.tsv", delimiter="\t")
genbank_mapping["Run"] = genbank_mapping["sra_run"]
genbank_mapping = genbank_mapping.drop_duplicates(subset="Run", keep="first")
genbank_mapping["name_state"] = genbank_mapping["genbank_name"].apply(lambda x: x.split("/")[2])

metadata_genbank = metadata_new.merge(genbank_mapping, on=["Run"], how="inner")
# metadata_new["name_state"] = "unknown"
# metadata_genbank = metadata_new

print(genbank_mapping["name_state"])
print(len(metadata_genbank))
display(metadata_genbank)

0        Texas
8        Texas
16       Texas
24       Texas
32       Texas
         ...  
37270       OH
37278       OH
37286       OH
37294       OH
37302       MT
Name: name_state, Length: 4103, dtype: object
3896


,Run,Assay Type,AvgSpotLen,Bases,BioProject,BioSample,BioSampleModel,Bytes,Center Name,Collection_Date,...,retraction_detection_date_utc,Genotype,seg_file,seg_seq_name,sra_run,seg,genbank_acc,genbank_seg,genbank_name,name_state
0,SRR28752447,WGS,241.29,86080323,PRJNA1102327,SAMN41019237,Viral,28164109,USDA-NVSL,2024,...,NaN,D1.3,SRR28752447_HA_cns.fa,Consensus_SRR28752447_HA_cns_threshold_0.5_qua...,SRR28752447,HA,PP752829.1,4,A/cattle/Texas/24-009108-005/2024,Texas
1,SRR28752448,WGS,250.30,75035343,PRJNA1102327,SAMN41019236,Viral,24547283,USDA-NVSL,2024,...,NaN,B3.13,SRR28752448_HA_cns.fa,Consensus_SRR28752448_HA_cns_threshold_0.5_qua...,SRR28752448,HA,PP752821.1,4,A/cattle/Texas/24-009108-004/2024,Texas
2,SRR28752449,WGS,146.61,59363690,PRJNA1102327,SAMN41019235,Viral,19686302,USDA-NVSL,2024,...,NaN,B3.13,SRR28752449_HA_cns.fa,Consensus_SRR28752449_HA_cns_threshold_0.5_qua...,SRR28752449,HA,PP752813.1,4,A/cattle/Texas/24-009108-003/2024,Texas
3,SRR28752450,WGS,251.31,119232569,PRJNA1102327,SAMN41019234,Viral,38846827,USDA-NVSL,2024,...,NaN,B3.13,SRR28752450_HA_cns.fa,Consensus_SRR28752450_HA_cns_threshold_0.5_qua...,SRR28752450,HA,PP752805.1,4,A/cattle/Texas/24-009108-002/2024,Texas
4,SRR28752453,WGS,144.71,37055919,PRJNA1102327,SAMN41019231,Viral,12938409,USDA-NVSL,2024,...,NaN,B3.13,SRR28752453_HA_cns.fa,Consensus_SRR28752453_HA_cns_threshold_0.5_qua...,SRR28752453,HA,PP752677.1,4,A/cattle/Texas/24-009088-001/2024,Texas
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3891,SRR32633088,WGS,145.11,67016467,PRJNA1102327,SAMN47290851,Viral,25388495,USDA-NVSL,2025,...,NaN,B3.13,SRR32633088_HA_cns.fa,Consensus_SRR32633088_HA_cns_threshold_0.5_qua...,SRR32633088,HA,PV456280.1,4,A/cat/OR/25-005913-003-original/2025,OR
3892,SRR32633089,WGS,148.14,140023511,PRJNA1102327,SAMN47290850,Viral,52157519,USDA-NVSL,2025,...,NaN,B3.13,SRR32633089_HA_cns.fa,Consensus_SRR32633089_HA_cns_threshold_0.5_qua...,SRR32633089,HA,PV456272.1,4,A/cat/OR/25-005800-002-original/2025,OR
3893,SRR32633090,WGS,148.49,83727737,PRJNA1102327,SAMN47290849,Viral,31619233,USDA-NVSL,2025,...,NaN,D1.3,SRR32633090_HA_cns.fa,Consensus_SRR32633090_HA_cns_threshold_0.5_qua...,SRR32633090,HA,PV456264.1,4,A/cat/OR/25-005800-001-original/2025,OR
3894,SRR32633093,WGS,148.10,105910536,PRJNA1102327,SAMN47290846,Viral,39757708,USDA-NVSL,2025,...,NaN,B3.2,SRR32633093_HA_cns.fa,Consensus_SRR32633093_HA_cns_threshold_0.5_qua...,SRR32633093,HA,PV457240.1,4,A/cattle/CA/25-005677-001-original/2025,CA


In [ ]:

metadata_genbank["Collection_Date_Specific"] = metadata_genbank["BioSample"].apply(lambda x: search_collection_date(x, metadata_genbank))
# metadata_genbank["Collection_Date_Specific"] = metadata_genbank["years"]

SAMN41019237
SAMN41019236
SAMN41019235
SAMN41019234
SAMN41019231
SAMN41019230
SAMN41019229
SAMN41019228
SAMN41019183
SAMN41019227
SAMN41019226
SAMN41019225


In [ ]:
# # Upload saved data
# os.chdir(temp_files)
# metadata_genbank = pd.read_csv("metadata_genbank.csv")

In [ ]:
# Save this so we don't have to do it again

os.chdir(temp_files)
metadata_genbank.to_csv("metadata_genbank.csv")

In [36]:

unique_animals_all = sort_animals_andersen(metadata_genbank)

# Flatten unique_animals
every_unique_animal = []
for animal in unique_animals_all:
    every_unique_animal.append(animal)

print(every_unique_animal)

unique_animals_set = list(set(every_unique_animal))
# animals_df = pd.DataFrame(columns=["avian", "cattle", "feline", "other_mammal", "human", "other"])
# animals_df["other"] = unique_animals_set # to sort

os.chdir(downloads)

animals_ref = pd.read_csv("animals_ref.csv")


# If animal not in ref1, put in ref2

common_animals = []
# Check if animals in unique_animals_set are in ref1
for animal in unique_animals_set:
    for col in animals_ref.columns:
        if animal in animals_ref[col].values and type(animal) == str:
            common_animals.append(animal)

print(common_animals)
print(len(common_animals))

different_animals = []
for animal in unique_animals_set:
    if animal not in common_animals:
        different_animals.append(animal)

print(different_animals)

# Add to dataframe
animals_df = animals_ref
# Make different_animals same length as dataframe, if shorter
if len(different_animals) < len(animals_df):
    number_of_times_to_add_nan = len(animals_df) - len(different_animals)
    for i in range(number_of_times_to_add_nan):
        different_animals.append(float('nan'))
# If longer, deal with that later

animals_df["new"] = (different_animals)

print(animals_df)

animals_df.to_csv("animals_ref_to_sort.csv")


['crow', 'goose', 'chicken', 'bald eagle', 'duck', 'great horned owl', 'mallard', 'sandhill crane', "cooper's hawk", 'red-tailed hawk', 'turkey', 'canada goose']
['crow', 'goose', 'chicken', 'bald eagle', 'duck', 'great horned owl', 'mallard', 'sandhill crane', "cooper's hawk", 'red-tailed hawk', 'turkey', 'canada goose']
12
[]
                      avian               cattle        feline   other_mammal  \
0          great_horned_owl            dairy_cow           cat         bobcat   
1              common_raven               cattle  domestic_cat    house_mouse   
2             cooper's_hawk  cattle milk product     feral_cat          skunk   
3              coopers_hawk                  NaN        feline         cougar   
4                   peafowl                  NaN  domestic-cat  geoffroys_cat   
..                      ...                  ...           ...            ...   
279           american crow                  NaN           NaN            NaN   
280  eurasian collared

In [37]:
# Get animals from animal reference
os.chdir(downloads)
animals_ref = pd.read_csv("animals_ref.csv")
fix_animals_andersen(metadata_genbank, animals_ref) # Get host type
metadata_genbank["years"] = metadata_genbank["Collection_Date"].apply(lambda x: str(x).split("-")[0]) # Get year only from collection date

In [38]:
for num, collection_date in enumerate(metadata_genbank["Collection_Date_Specific"]):
    if collection_date != collection_date: # If nan
        metadata_genbank.loc[num, "Collection_Date_Specific"] = metadata_genbank.loc[num, "years"]
    else: # If actual date
        if len(str(collection_date)) == 4: # If it's a year
            # print("caught")
            metadata_genbank.loc[num, "Collection_Date_Specific"] = collection_date
        else:
            parsed_date = dateutil.parser.parse(collection_date)
            date = parsed_date.strftime("%Y-%m-%d") # Make sure it doesn't default to today, if just a year
            metadata_genbank.loc[num, "Collection_Date_Specific"] = date

# Make names

names = ">A/" + metadata_genbank["Host"] + "/" + metadata_genbank["name_state"] + "/" + metadata_genbank["isolate"] + "/" + metadata_genbank["years"].apply(lambda x: str(x)) + "|H5N1|" + metadata_genbank["Collection_Date_Specific"].apply(lambda x: str(x)) + "|" + metadata_genbank["Host_Type"] + "|" + metadata_genbank["Genotype"]

metadata_genbank["Name"] = names

print(metadata_genbank["years"])

metadata_genbank.to_csv("metadata_genbank_named.csv")

# display(metadata_genbank)

0      2025
1      2025
2      2025
3      2025
4      2025
       ... 
273    2025
274    2025
275    2025
276    2025
277    2025
Name: years, Length: 278, dtype: object


In [ ]:
# Make fasta files

fasta_folder = originals + "avian-influenza/fasta/"

os.chdir(fasta_folder)

pairs = []
fasta_files = {}

for genotype in ["B3.13, D1.1"]:
    for segment in ["PB2", "PB1", "PA", "NS", "NP", "NA", "MP", "HA"]:
        pair = genotype + "_" + segment
        pairs.append(pair)

for pair in pairs:
    fasta_files[pair] = [] # List to hold fasta files

for run in metadata_genbank["Run"].values: # For each run 
    for dirpath, dirs, files in os.walk(fasta_folder): # Find the fasta file
        for file in files:
            file_name = os.path.join(dirpath, file) # Get file name
            # print(file_name)
            if run in file_name: # Note that there will be ~8 files total with that run name
                # Make a fasta file and put it in the list
                with open(file_name) as f:
                    lines = f.readlines()
                    sequence = lines[1] 
                    # Each run/segment pair has one sequence -- it's placed into a file with other run/segment pairs with the same segment and genotype
                    header = metadata_genbank[metadata_genbank["Run"] == run].loc[:, "Name"].values[0]
                    genotype = metadata_genbank[metadata_genbank["Run"] == run].loc[:, "Genotype"].values[0]
                    # print(header)
                    # print(genotype)
                    # break 
                    segment = file_name.split("_")[-2]
                    # Find the pair that corresponds to 
                    pair_name = genotype + "_" + segment
                    this_specific_fasta = []
                    for pair in pairs:
                        # print(pair)
                        # print(pair_name)
                        if pair_name == pair:
                            this_specific_fasta.append(header)
                            this_specific_fasta.append(sequence)
                            fasta_files[pair].append(this_specific_fasta)
                f.close()

In [ ]:
# Create fasta files 

date = "2025-04-14"

os.chdir(temp_files)

for pair in fasta_files.keys():
    output_path = temp_files + pair + "_andersen_" + date + ".fasta" 

    output_file = open(output_path, "w")
    for item in fasta_files[pair]:
        # for item in item:
        # item = fasta_files[pair]
        try:
            name = str(item[0].values[0]) # See if this is one we didn't have a collection date for
        except:
            name = str(item[0])
        print(name)
        # First is header, second is sequence
        # print(value)
        output_file.write(name + "\n")
        output_file.write(item[1])
    output_file.close()

>A/TURKEY/unknown/25-002414-001/2025|H5N1|2025|avian|D1.3
>A/CHICKEN/unknown/25-002299-002/2025|H5N1|2025|avian|D1.3
>A/CHICKEN/unknown/25-002299-001/2025|H5N1|2025|avian|D1.3
>A/TURKEY/unknown/25-002298-002/2025|H5N1|2025|avian|D1.3
>A/TURKEY/unknown/25-002298-001/2025|H5N1|2025|avian|D1.3
>A/TURKEY/unknown/25-002297-004/2025|H5N1|2025|avian|D1.3
>A/TURKEY/unknown/25-002297-003/2025|H5N1|2025|avian|D1.3
>A/TURKEY/unknown/25-002297-002/2025|H5N1|2025|avian|D1.3
>A/TURKEY/unknown/25-002297-001/2025|H5N1|2025|avian|D1.3
>A/TURKEY/unknown/25-002295-004/2025|H5N1|2025|avian|D1.3
>A/TURKEY/unknown/25-002295-003/2025|H5N1|2025|avian|D1.3
>A/TURKEY/unknown/25-002295-002/2025|H5N1|2025|avian|D1.3
>A/CHICKEN/unknown/25-002277-002/2025|H5N1|2025|avian|D1.3
>A/CHICKEN/unknown/25-002277-001/2025|H5N1|2025|avian|D1.3
>A/CHICKEN/unknown/25-002275-001/2025|H5N1|2025|avian|D1.3
>A/TURKEY/unknown/25-002274-002/2025|H5N1|2025|avian|D1.3
>A/TURKEY/unknown/25-002274-001/2025|H5N1|2025|avian|D1.3
>A/CHICKE

In [ ]:
# De-duplication 

# Gisaid 

gisaid = downloads + "GISAID_Complete_Fasta_Files/" #04-01-2025--04-14-2025/"

os.chdir(gisaid)



In [43]:
dfs_gisaid = create_dataframes(gisaid)
print(dfs_gisaid["B3.13_HA"][0])

    isolate_partial                                        full_header  \
0                P-  >A/dairy_cow/Idaho/W241290019-18-P/2024|H5N1|2...   
1                P-  >A/dairy_cow/Idaho/W241290019-19-P/2024|H5N1|2...   
2      W241220059-8  >A/dairy_cow/Idaho/W241220059-8/2024|H5N1|2024...   
3      W241220059-9  >A/dairy_cow/Idaho/W241220059-9/2024|H5N1|2024...   
4     W240870066-26  >A/dairy_cow/Idaho/W240870066-26/2024|H5N1|202...   
..              ...                                                ...   
202      000600-002  >A/dairy_cow/California/25_000600-002/2024|H5N...   
203      000590-002  >A/dairy_cow/California/25_000590-002/2024|H5N...   
204      005209-001  >A/dairy_cow/California/25_005209-001/2024|H5N...   
205      004920-005  >A/dairy_cow/California/25_004920-005/2024|H5N...   
206      004920-004  >A/dairy_cow/California/25_004920-004/2024|H5N...   

                                              sequence  
0    atggagaacatagtactacttcttgcaatagttagccttgttaaaa...

In [44]:
# Do the same with Andersen 

dfs_andersen = create_dataframes(temp_files)

In [ ]:
# print(dfs_andersen["D1.3_HA"][0])

    isolate_partial                                        full_header  \
0        002414-001  >A/TURKEY/unknown/25-002414-001/2025|H5N1|2025...   
1        002299-002  >A/CHICKEN/unknown/25-002299-002/2025|H5N1|202...   
2        002299-001  >A/CHICKEN/unknown/25-002299-001/2025|H5N1|202...   
3        002298-002  >A/TURKEY/unknown/25-002298-002/2025|H5N1|2025...   
4        002298-001  >A/TURKEY/unknown/25-002298-001/2025|H5N1|2025...   
..              ...                                                ...   
273      005210-003  >A/CHICKEN/unknown/25-005210-003/2025|H5N1|202...   
274      005210-002  >A/CHICKEN/unknown/25-005210-002/2025|H5N1|202...   
275      005210-001  >A/CHICKEN/unknown/25-005210-001/2025|H5N1|202...   
276      010298-001  >A/SANDHILL CRANE/unknown/25-010298-001/2025|H...   
277      010466-001  >A/CHICKEN/unknown/25-010466-001/2025|H5N1|202...   

                                              sequence  
0    ATGGAAAACATAGTACTTCTTCTTGCAATAATTAGCCTTGTTAAAA...

In [ ]:
# Merge dataframes and drop duplicates

full_dfs = defaultdict(list)
for key in dfs_andersen.keys():
    dataframes = dfs_andersen[key]
    for i, df in enumerate(dataframes):
        print(i)
        try:
            full_df = df.merge(dfs_gisaid[key][i], how="outer")
            # print(full_df)
            full_df = full_df.drop_duplicates(subset=["isolate_partial"])
            full_dfs[key].append(full_df)
        except:
            print("Failed to merge dataframes in ", key)



In [ ]:
# If none in one database, only use the other and drop duplicates

full_dfs = defaultdict(list)
for key in dfs_andersen.keys():
    print(key)
# for key in ["D1.3"]:
    dataframes = dfs_andersen[key]
    for i, df in enumerate(dataframes):
        print(i)
        try:
            # full_df = df.merge(dfs_gisaid[key][i], how="outer")
            # print(full_df)
            full_df = full_df.drop_duplicates(subset=["isolate_partial"])
            full_dfs[key].append(full_df)
        except:
            print("Failed to merge dataframes in ", key)

In [56]:
print(full_dfs["D1.3_HA"]) #[0])

[]


In [ ]:
# Create fasta files 

os.chdir(complete_files)
for pair in full_dfs.keys():
    output_path = complete_files + pair + "_combined_" + date + ".fasta" 

    output_file = open(output_path, "w")
    for item in full_dfs[pair]:
        # for item in item:
        # item = fasta_files[pair]
        for index, row in item.iterrows():
            name = item.loc[index, "full_header"]
            sequence = item.loc[index, "sequence"]
        # print(name)
        # First is header, second is sequence
        # print(value)
            output_file.write(name)
            output_file.write(sequence)
    output_file.close()